# Temperature Trajectories by TTM Group Across Datasets

This notebook generates a supplementary figure showing hourly mean temperature trajectories
for TTM vs no-TTM patients across the four datasets used in the manuscript:

- **HYPERION** (RCT, France) — TTM from randomization arm, per-day temperature columns
- **eICU-CRD** (US multicenter) — TTM from treatment string + temperature derived; temperature from nurseCharting
- **MIMIC-IV** (BIDMC) — TTM from cooling-device itemid 225052; temperature from chartevents itemid 223761/223762
- **PMAP / OSLER** (JHU) — TTM from Fahrenheit temperature rule; temperature from Epic flowsheet meas_id 6

## How to run this
Each dataset section is self-contained and can be run on the machine where that dataset lives.
Each section produces a small aggregated CSV (`ts_<dataset>.csv`) with group-hour summaries — no PHI.
Once all four CSVs exist, run the final plotting section to produce the figure.

**Paths you may need to edit** are collected in the config cell below.



## 0. Config and imports

In [2]:
# Paths — edit per machine you run this on
EICU_DB         = '/projects/LCICM/eICU/'                  # contains nurseCharting.csv
EICU_PREDICTORS = './eICU/eICUPredictorsDiag.csv'              # your saved feature table with 'both_hypothermia'
EICU_IDS        = './eICU/patient_ids.csv'                        # same file you already use

MIMIC_DB         = '/projects/LCICM/'                      # contains mimic-iv-2.2/icu/chartevents.csv
MIMIC_CA_TIME    = './mimiciv/CA_time.csv'                           # cardiac arrest reference time
MIMIC_PREDICTORS = './mimiciv/MIMIC_Predictors2.csv'            # your saved feature table with 'hypothermia'

PMAP_DB          = '/projects/LCICM/ACCMPMAP/'             # contains Isalis/Isalis_Flowsheet*.csv
PMAP_PREDICTORS  = './pmap/PMAP_Predictors2.csv'             # your saved feature table with 'hypothermia'
PMAP_OHCA        = './pmap/ohca.csv'                            # for admission time reference

# HYPERION_CSV     = './hyperion.csv'                        # the trial CSV with J0_TEMP, J1_TEMP, BIO_TEMP, groupe
# HYPERION_GROUP_COL = 'groupe'                              # 1 = TTM, 2 = normothermia (confirm with Lascarrou)

# Analysis parameters
MAX_HOURS  = 72     # analysis horizon
BIN_HOURS  = 1      # hourly bins
OUTPUT_DIR = './timeseries_output'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


## 1. Shared utilities

In [4]:
def to_celsius(series):
    """Convert to numeric °C, assuming values >50 are Fahrenheit."""
    s = pd.to_numeric(series, errors='coerce')
    is_f = s > 50
    s = s.where(~is_f, (s - 32) * 5.0 / 9.0)
    return s


def bin_hourly(df, id_col, ttm_col, time_min_col, temp_c_col, max_hours=MAX_HOURS):
    """Per-patient, per-hour mean temperature over [0, max_hours]."""
    d = df[[id_col, ttm_col, time_min_col, temp_c_col]].copy()
    d = d.rename(columns={id_col:'id', ttm_col:'ttm',
                          time_min_col:'t_min', temp_c_col:'temp_c'})
    d['temp_c'] = pd.to_numeric(d['temp_c'], errors='coerce')
    d['t_min']  = pd.to_numeric(d['t_min'], errors='coerce')
    d = d.dropna(subset=['temp_c','t_min'])
    d = d[(d['temp_c']>=25) & (d['temp_c']<=42)]
    d = d[(d['t_min']>=0) & (d['t_min']<=max_hours*60)]
    d['hour'] = (d['t_min']//60).astype(int)
    return (d.groupby(['id','ttm','hour'], as_index=False)['temp_c'].mean())


def summarize(hourly_long, dataset):
    """Aggregate patient-hours to group-hour summary (mean, median, IQR, n)."""
    agg = (hourly_long.groupby(['ttm','hour'])['temp_c']
           .agg(mean='mean',
                median='median',
                q25=lambda x: np.nanpercentile(x, 25),
                q75=lambda x: np.nanpercentile(x, 75),
                n='count')
           .reset_index())
    agg.insert(0,'dataset',dataset)
    return agg


## 2. HYPERION

HYPERION stores temperatures as per-day columns (`J0_TEMP`, `J1_TEMP`, ... `J5_TEMP` or similar).
We expand each daily column into a long row with hour = day * 24. If your data has more
granular temperature timestamps in a separate file, substitute that file here instead.

**TTM label**: `groupe == 1` is the TTM arm. Confirm with the trial codebook before publishing.


In [ ]:
def run_hyperion(csv_path=HYPERION_CSV, group_col=HYPERION_GROUP_COL, id_col='SUBJID'):
    df = pd.read_csv(csv_path)
    day_cols = [c for c in df.columns if c.split('_')[0].startswith(('J','V'))
                and c.endswith('_TEMP')]
    print(f'Found HYPERION daily temp columns: {day_cols}')
    records = []
    for c in day_cols:
        try:
            day = int(c.split('_')[0][1:])
        except ValueError:
            continue
        hour = day * 24
        sub = df[[id_col, group_col, c]].copy()
        sub['temp_c'] = to_celsius(sub[c])
        sub['ttm']    = (sub[group_col] == 1).astype(int)
        sub['hour']   = hour
        sub = sub[[id_col, 'ttm', 'hour', 'temp_c']]
        sub = sub.rename(columns={id_col:'id'})
        records.append(sub)
    long = pd.concat(records, ignore_index=True).dropna(subset=['temp_c'])
    long = long[(long['temp_c']>=25) & (long['temp_c']<=42) & (long['hour']<=MAX_HOURS)]
    summary = summarize(long, 'HYPERION')
    out_path = f'{OUTPUT_DIR}/ts_hyperion.csv'
    summary.to_csv(out_path, index=False)
    print(f'Wrote {out_path} ({len(summary)} rows)')
    return summary

# summary_hyp = run_hyperion()   # uncomment to run


## 3. eICU-CRD

Temperature comes from `nurseCharting.csv` where `nursingchartcelltypevalname == "Temperature (C)"`.
Time offset is `nursingchartentryoffset` (minutes from ICU admission).

**TTM label**: the `both_hypothermia` flag in your saved `myPredictorsDf` (treatment-string OR
temperature-derived, per eICU__1_.ipynb). This matches the definition in the paper.


In [6]:
def run_eicu(db=EICU_DB, predictors_csv=EICU_PREDICTORS, ids_csv=EICU_IDS):
    preds = pd.read_csv(predictors_csv, usecols=['patientunitstayid','both_hypothermia'])
    preds = preds.rename(columns={'both_hypothermia':'ttm'})

    # If you have an IDs file, restrict; otherwise use all predictor IDs
    try:
        ids = pd.read_csv(ids_csv).patientunitstayid.unique()
        preds = preds[preds.patientunitstayid.isin(ids)]
    except FileNotFoundError:
        pass

    print(f'eICU analytic cohort: n={preds.patientunitstayid.nunique()}')

    chunks = []
    file_path = db + 'nurseCharting.csv'
    for chunk in tqdm(pd.read_csv(file_path, chunksize=1_000_000,
                                  usecols=['patientunitstayid',
                                           'nursingchartentryoffset',
                                           'nursingchartcelltypevalname',
                                           'nursingchartvalue'])):
        chunk = chunk[chunk['nursingchartcelltypevalname']=='Temperature (C)']
        chunk = chunk[chunk['patientunitstayid'].isin(preds.patientunitstayid)]
        if len(chunk):
            chunks.append(chunk)
    raw = pd.concat(chunks, ignore_index=True)
    raw = raw.merge(preds, on='patientunitstayid', how='inner')
    raw['temp_c'] = to_celsius(raw['nursingchartvalue'])

    hourly = bin_hourly(raw, 'patientunitstayid', 'ttm',
                        'nursingchartentryoffset', 'temp_c')
    summary = summarize(hourly, 'eICU')
    out_path = f'{OUTPUT_DIR}/ts_eicu.csv'
    summary.to_csv(out_path, index=False)
    print(f'Wrote {out_path} ({len(summary)} rows)')
    return summary

summary_eicu = run_eicu()   # uncomment to run


eICU analytic cohort: n=3172


71it [00:53,  1.34it/s]

## 4. MIMIC-IV

Temperature itemids in MIMIC-IV:
- `223761` — Temperature Fahrenheit
- `223762` — Temperature Celsius

Time offset is `chartoffset` (minutes from arrest reference time, per Feature_extraction.ipynb
convention where chartoffset is computed as `charttime - ca_time`).

**TTM label**: `hypothermia` flag in your saved `myPredictorsDf`, derived from cooling device
itemid 225052 being "On" continuously for ≥720 minutes.


In [ ]:
MIMIC_TEMP_ITEMIDS = [223761, 223762]

def run_mimic(db=MIMIC_DB, predictors_csv=MIMIC_PREDICTORS, ca_time_csv=MIMIC_CA_TIME):
    preds = pd.read_csv(predictors_csv, usecols=['subject_id','hypothermia'])
    preds = preds.rename(columns={'hypothermia':'ttm'})

    # CA reference time for computing chartoffset
    ca_time = pd.read_csv(ca_time_csv)
    ca_time['time'] = pd.to_datetime(ca_time['time'], errors='coerce')
    print(f'MIMIC analytic cohort: n={preds.subject_id.nunique()}')

    chunks = []
    file_path = db + 'mimic-iv-2.2/icu/chartevents.csv'
    usecols = ['subject_id','charttime','itemid','valuenum','value']
    for chunk in tqdm(pd.read_csv(file_path, chunksize=1_000_000, usecols=usecols)):
        chunk = chunk[chunk['itemid'].isin(MIMIC_TEMP_ITEMIDS)]
        chunk = chunk[chunk['subject_id'].isin(preds.subject_id)]
        if len(chunk):
            chunks.append(chunk)
    raw = pd.concat(chunks, ignore_index=True)
    raw = raw.merge(ca_time[['subject_id','time']], on='subject_id', how='left')
    raw['charttime']  = pd.to_datetime(raw['charttime'], errors='coerce')
    raw['chartoffset'] = (raw['charttime'] - raw['time']).dt.total_seconds() / 60
    raw = raw[raw['chartoffset']>=0]
    raw = raw.merge(preds, on='subject_id', how='inner')

    # valuenum should already be numeric; fall back to value if missing
    raw['temp_c'] = to_celsius(raw['valuenum'].fillna(raw['value']))

    hourly = bin_hourly(raw, 'subject_id', 'ttm', 'chartoffset', 'temp_c')
    summary = summarize(hourly, 'MIMIC-IV')
    out_path = f'{OUTPUT_DIR}/ts_mimic.csv'
    summary.to_csv(out_path, index=False)
    print(f'Wrote {out_path} ({len(summary)} rows)')
    return summary

summary_mimic = run_mimic()   # uncomment to run


## 5. PMAP (JHU OSLER/Epic)

Temperature `meas_id` values, per Feature_extraction__1_.ipynb:
- `6`
- `304301490`

Time offset is `meas_offset` in minutes from hospital admission
(computed from `recorded_time - hosp_admsn_time`).

**TTM label**: `hypothermia` flag in your saved `myPredictorsDf`, derived from average first-24h
temperature <96.8°F (≈36°C) with minimum ≥77°F (≈25°C).


In [ ]:
PMAP_TEMP_MEAS_IDS = [6, 304301490]

def read_flowsheet(file_path, chunk_size=1_000_000, head=None):
    """Matches the read_flowsheet function in Feature_extraction__1_.ipynb."""
    cols = ['osler_id','pat_enc_csn_id','inpatient_data_id','recorded_time','meas_value',
            'meas_comment','meas_id','meas_row_type_c','meas_val_type_c','meas_template_id',
            'meas_fsd_id','meas_line','meas_occurance','rw_meas_id',
            'ip_lda_id','rw_row_type_c','rw_val_type_c','meas_taken_user_id']
    chunks = []
    for chunk in pd.read_csv(file_path, chunksize=chunk_size, header=head):
        chunk.columns = cols
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)


def run_pmap(db=PMAP_DB, predictors_csv=PMAP_PREDICTORS, ohca_csv=PMAP_OHCA):
    preds = pd.read_csv(predictors_csv, usecols=['osler_id','hypothermia'])
    preds = preds.rename(columns={'hypothermia':'ttm'})
    ohca = pd.read_csv(ohca_csv, usecols=['osler_id','hosp_admsn_time'])

    print(f'PMAP analytic cohort: n={preds.osler_id.nunique()}')

    # Read all three flowsheet parts (per your notebook)
    parts = []
    for fname in ['Isalis/Isalis_Flowsheet_part1.csv',
                  'Isalis/Isalis_Flowsheet_part2.csv',
                  'Isalis/Isalis_Flowsheetdata.csv']:
        p = db + fname
        print(f'Reading {p}')
        parts.append(read_flowsheet(p))
    flow = pd.concat(parts, ignore_index=True)
    flow = flow[flow['osler_id'].isin(preds.osler_id)]
    flow = flow[flow['meas_id'].isin(PMAP_TEMP_MEAS_IDS)]

    # compute meas_offset in minutes from hospital admission
    flow = flow.merge(ohca, on='osler_id', how='left')
    flow['recorded_time']   = pd.to_datetime(flow['recorded_time'], errors='coerce')
    flow['hosp_admsn_time'] = pd.to_datetime(flow['hosp_admsn_time'], errors='coerce')
    flow['meas_offset']     = (flow['recorded_time'] - flow['hosp_admsn_time']).dt.total_seconds() / 60
    flow = flow[flow['meas_offset'] >= 0]

    flow = flow.merge(preds, on='osler_id', how='inner')
    flow['temp_c'] = to_celsius(flow['meas_value'])

    hourly = bin_hourly(flow, 'osler_id', 'ttm', 'meas_offset', 'temp_c')
    summary = summarize(hourly, 'PMAP')
    out_path = f'{OUTPUT_DIR}/ts_pmap.csv'
    summary.to_csv(out_path, index=False)
    print(f'Wrote {out_path} ({len(summary)} rows)')
    return summary

summary_pmap = run_pmap()   # uncomment to run


## 6. Combine and plot

Run this after the four summary CSVs exist. It produces a 2x2 panel figure ready for the
supplement. The plot format matches typical ICM supplement style.


In [ ]:
import matplotlib.pyplot as plt

def plot_trajectories(output_dir=OUTPUT_DIR,
                      out_png='supp_fig_temperature_trajectories.png',
                      out_pdf='supp_fig_temperature_trajectories.pdf'):
    files = {
        # 'HYPERION': f'{output_dir}/ts_hyperion.csv',
        'eICU':     f'{output_dir}/ts_eicu.csv',
        'PMAP':     f'{output_dir}/ts_pmap.csv',
        'MIMIC-IV': f'{output_dir}/ts_mimic.csv',
    }
    loaded = {}
    for name, path in files.items():
        try:
            loaded[name] = pd.read_csv(path)
        except FileNotFoundError:
            print(f'  (missing: {path})')

    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True, sharey=True)
    colors = {0: '#2c5282', 1: '#c53030'}   # blue no-TTM, red TTM
    labels = {0: 'No TTM', 1: 'TTM'}

    panel_order = ['HYPERION', 'eICU', 'PMAP', 'MIMIC-IV']
    for ax, name in zip(axes.ravel(), panel_order):
        if name not in loaded:
            ax.set_title(f'{name} (no data)', fontsize=10)
            ax.axis('off')
            continue
        sub = loaded[name]
        for ttm_val in [0, 1]:
            g = sub[sub['ttm']==ttm_val].sort_values('hour')
            if g.empty:
                continue
            ax.plot(g['hour'], g['mean'], color=colors[ttm_val], lw=2,
                    label=f'{labels[ttm_val]} (max n={int(g["n"].max())})')
            ax.fill_between(g['hour'], g['q25'], g['q75'],
                            color=colors[ttm_val], alpha=0.18)
        ax.axhline(33, ls=':', color='#666', lw=0.8)
        ax.axhline(36, ls=':', color='#666', lw=0.8)
        ax.set_title(name, fontsize=11, fontweight='bold')
        ax.set_ylim(30, 40)
        ax.set_xlim(0, MAX_HOURS)
        ax.legend(loc='lower right', fontsize=8, frameon=False)
        ax.grid(alpha=0.25)

    for ax in axes[-1]:
        ax.set_xlabel('Hours from admission / randomization')
    for ax in axes[:,0]:
        ax.set_ylabel('Temperature (°C)')

    fig.suptitle('Temperature trajectories by TTM group across datasets',
                 fontsize=12, y=1.00)
    fig.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    fig.savefig(out_pdf, bbox_inches='tight')
    print(f'Wrote {out_png} and {out_pdf}')
    return fig

fig = plot_trajectories()


## 7. Optional: companion table of per-group minimum temperatures

Quantifies the clinical construct-validity argument numerically. Useful as Supplementary Table S20
or as a 2-row note under the figure.


In [ ]:
def nadir_table(output_dir=OUTPUT_DIR):
    rows = []
    for name, path in [#('HYPERION', 'ts_hyperion.csv'),
                       ('eICU',     'ts_eicu.csv'),
                       ('PMAP',     'ts_pmap.csv'),
                       ('MIMIC-IV', 'ts_mimic.csv')]:
        try:
            d = pd.read_csv(f'{output_dir}/{path}')
        except FileNotFoundError:
            continue
        for ttm_val in [0, 1]:
            sub = d[d['ttm']==ttm_val]
            if sub.empty:
                continue
            nadir_row = sub.loc[sub['mean'].idxmin()]
            rows.append({
                'Dataset':      name,
                'Group':        'TTM' if ttm_val==1 else 'No TTM',
                'Nadir (°C)':   round(nadir_row['mean'], 2),
                'Hour of nadir': int(nadir_row['hour']),
                'IQR at nadir': f"[{nadir_row['q25']:.2f}, {nadir_row['q75']:.2f}]",
                'n at nadir':   int(nadir_row['n']),
            })
    out = pd.DataFrame(rows)
    out.to_csv(f'{output_dir}/nadir_table.csv', index=False)
    print(out.to_string(index=False))
    return out

nadir_tbl = nadir_table()
